In [1]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.agents import create_agent
from langchain.tools import tool

load_dotenv()

C:\Users\vijay\AppData\Local\Temp\ipykernel_30908\624781542.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


True

In [2]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

### Loading Data

In [3]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")

### Data Ingestion

In [4]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")
documents = loader.load()
print(documents)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDuring probati

### Split Document into smaller chunks

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=50, separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print(chunks)
print(chunks[0].page_content)
print(chunks[0].metadata)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

### Converting chunks into Embeddings

In [6]:
embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print(f"Embedding model is ready: {embeddings_model.model_name}")

Embedding model is ready: jina-embeddings-v2-base-en


### Store embeddings in a vector DB

In [7]:
vector_store = FAISS.from_documents(chunks, embeddings_model)

print(f"Chunks are stored: {vector_store.index.ntotal}")

Chunks are stored: 9


In [ ]:
## Similarity search
test_query = "How many sick leave employees get"

top_matches = vector_store.similarity_search(test_query, k=2)

for i, match in enumerate(top_matches, start=1):
    print(f"------Match {i}---------")
    print(match.page_content)
    print()

------Match 1---------
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

------Match 2---------
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



### Data Retrieval

In [8]:
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.9)

test_response = llm.invoke("Hey who are you, explain in poetry manner in 2 lines")
test_response.content

'I am a wandering echo of code, a mind that learns from every line—  \nA digital muse, weaving words into light, forever bound to answer yours in rhyme.'

### AI Agents
- LLM : Brain
- Tool : Super power
- Memory 

In [9]:
retriever=vector_store.as_retriever(search_kwargs={"k":3})

@tool
def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks=retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)



In [10]:

hr_assistant = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt="""
    You are a friendly HR assistant,
    Always use the search_hr_policy tool to look up any hr related policy facts
    before answering, if the answer is not in search_results, say You Don't Know instead of guessing
""",
)

In [11]:
def ask_hr_assistant(question:str)->str:
    """Send question to the RAG agent and print a nicely formatted answer."""
    print("="*60)
    print("QUESTION:",question)
    print("-"*60)

    response=hr_assistant.invoke({"messages":[{"role":"user","content":question}]})
    answer=response["messages"][-1].content

    print("ANSWER:",answer)
    print("="*60)
    print()
    return answer

In [ ]:
response=hr_assistant.invoke({"messages":[{"role":"user","content":"Tell me about reimbursement policy"}]})
print(response["messages"][-1].content)

Sure! I’m your friendly HR assistant, here to help you navigate company policies, answer questions about leave, work‑from‑home arrangements, probation periods, notice requirements, reimbursements, the code of conduct, holidays, and the exit process. Think of me as a quick, reliable resource for anything HR‑related—just ask, and I’ll look up the official policy to give you an accurate answer. If there’s something outside my scope, I’ll let you know. How can I assist you today?
